# 인구 데이터 전처리 및 통합 파이프라인

분당구 30m 격자 기준 유동/직장/방문/서비스(주중·주말) 인구 데이터를 전처리하고 하나의 마스터 테이블로 통합한다.

## 1. 유동인구 전처리 (06번 성연령별 유동인구)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geoband.API import *

# 격자 데이터
gdf = gpd.read_file('04._격자_30mx30m.geojson')

# 06번 성연령별 유동인구 데이터
GetCompasData('SBJ_2606_002', '6', '06._성연령별_유동인구.csv')
df = pd.read_csv('06._성연령별_유동인구.csv')

In [ ]:
geometry = [
    Point(xy)
    for xy in zip(df['lon'], df['lat'])
]

pop_gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs='EPSG:4326'
)

In [ ]:
if gdf.crs != pop_gdf.crs:
    gdf = gdf.to_crs(pop_gdf.crs)

In [ ]:
joined_df = gpd.sjoin(
    pop_gdf,
    gdf[['grid_id', 'geometry']],
    how='left',
    predicate='within'
)

result_df = joined_df.drop(
    columns=['geometry', 'index_right'],
    errors='ignore'
).copy()

In [ ]:
df_summer = result_df[
    result_df['STD_YM'].astype(str).str.contains('202406|202407|202408|202409', regex=True)
    & result_df['grid_id'].notna()
].copy()

In [ ]:
pop_cols = [
    col for col in df_summer.columns
    if 'pop' in col
]

In [ ]:
grid_summer_avg = (
    df_summer
    .groupby('grid_id')[pop_cols]
    .mean()
    .reset_index()
)

In [ ]:
month_counts = (
    df_summer
    .groupby('grid_id')['STD_YM']
    .nunique()
    .reset_index(name='matched_months_cnt')
)

grid_summer_avg = grid_summer_avg.merge(
    month_counts,
    on='grid_id',
    how='left'
)

In [ ]:
final_summer_gdf = gdf.merge(
    grid_summer_avg,
    on='grid_id',
    how='left'
)

In [ ]:
final_summer_gdf.to_csv(
    'summer_flow_population.csv',
    index=False,
    encoding='utf-8-sig'
)

final_summer_gdf.to_file(
    'summer_flow_population.shp',
    encoding='utf-8'
)

## 2. 직장인구 전처리 (07번 시간대별 직장인구)

In [ ]:
## 01: 라이브러리 및 데이터 불러오기

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geoband.API import *

# 07번 직장인구 데이터 다운로드
GetCompasData('SBJ_2606_002', '7', '07._시간대별_직장인구.csv')

# 직장인구 데이터
df_worker = pd.read_csv('07._시간대별_직장인구.csv')

# 30m × 30m 격자 데이터
gdf = gpd.read_file('04._격자_30mx30m.geojson')

In [ ]:
## 02: 직장인구 좌표 → GeoDataFrame 변환

geometry = [
    Point(xy)
    for xy in zip(df_worker['lon'], df_worker['lat'])
]

worker_gdf = gpd.GeoDataFrame(
    df_worker,
    geometry=geometry,
    crs='EPSG:4326'
)

In [ ]:
## 03: 좌표계 통일

if gdf.crs != worker_gdf.crs:
    gdf = gdf.to_crs(worker_gdf.crs)

In [ ]:
## 04: 직장인구 Point와 격자 공간조인

joined_worker_df = gpd.sjoin(
    worker_gdf,
    gdf[['grid_id', 'geometry']],
    how='left',
    predicate='within'
)

result_worker_df = joined_worker_df.drop(
    columns=['geometry', 'index_right'],
    errors='ignore'
).copy()

In [ ]:
## 05: 2024년 6~9월 데이터 추출

df_worker_summer_all = result_worker_df[
    result_worker_df['STD_YM'].astype(str).str.contains(
        '202406|202407|202408|202409',
        regex=True
    )
    & result_worker_df['grid_id'].notna()
].copy()

In [ ]:
## 06: 직장인구 시간대 컬럼 추출

tmst_cols = [
    col
    for col in df_worker_summer_all.columns
    if col.startswith('TMST_')
]

In [ ]:
## 07: grid_id별 6~9월 평균 직장인구 계산

grid_worker_summer_avg = (
    df_worker_summer_all
    .groupby('grid_id')[tmst_cols]
    .mean()
    .reset_index()
)

In [ ]:
## 08: 격자별 데이터가 존재하는 월 수 계산

month_counts = (
    df_worker_summer_all
    .groupby('grid_id')['STD_YM']
    .nunique()
    .reset_index(name='matched_months_cnt')
)

grid_worker_summer_avg = grid_worker_summer_avg.merge(
    month_counts,
    on='grid_id',
    how='left'
)

In [ ]:
## 09: 격자 GeoDataFrame과 직장인구 데이터 결합

final_worker_summer_gdf = gdf.merge(
    grid_worker_summer_avg,
    on='grid_id',
    how='left'
)

In [ ]:
## 10: 최종 직장인구 데이터 저장

final_worker_summer_gdf.to_csv(
    'final_worker_summer_gdf.csv',
    index=False,
    encoding='utf-8-sig'
)

final_worker_summer_gdf.to_file(
    'final_worker_summer_gdf.shp',
    encoding='utf-8'
)

## 3. 방문인구 전처리 (08번 시간대별 방문인구)

In [ ]:
## 01: 라이브러리 및 데이터 불러오기

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geoband.API import *

# 08번 방문인구 데이터 다운로드
GetCompasData('SBJ_2606_002', '8', '08._시간대별_방문인구.csv')

# 방문인구 데이터
df_visitor = pd.read_csv('08._시간대별_방문인구.csv')

# 30m × 30m 격자 데이터
gdf = gpd.read_file('04._격자_30mx30m.geojson')

In [ ]:
## 02: 방문인구 좌표 → GeoDataFrame 변환

geometry = [
    Point(xy)
    for xy in zip(df_visitor['lon'], df_visitor['lat'])
]

visitor_gdf = gpd.GeoDataFrame(
    df_visitor,
    geometry=geometry,
    crs='EPSG:4326'
)

In [ ]:
## 03: 좌표계 통일

if gdf.crs != visitor_gdf.crs:
    gdf = gdf.to_crs(visitor_gdf.crs)

In [ ]:
## 04: 방문인구 Point와 격자 공간조인

joined_visitor_df = gpd.sjoin(
    visitor_gdf,
    gdf[['grid_id', 'geometry']],
    how='left',
    predicate='within'
)

result_visitor_df = joined_visitor_df.drop(
    columns=['geometry', 'index_right'],
    errors='ignore'
).copy()

In [ ]:
## 05: 2024년 6~9월 데이터 추출

df_visitor_summer_all = result_visitor_df[
    result_visitor_df['STD_YM'].astype(str).str.contains(
        '202406|202407|202408|202409',
        regex=True
    )
    & result_visitor_df['grid_id'].notna()
].copy()

In [ ]:
## 06: 방문인구 시간대 컬럼 추출

tmst_cols = [
    col
    for col in df_visitor_summer_all.columns
    if col.startswith('TMST_')
]

In [ ]:
## 07: grid_id별 6~9월 평균 방문인구 계산

grid_visitor_summer_avg = (
    df_visitor_summer_all
    .groupby('grid_id')[tmst_cols]
    .mean()
    .reset_index()
)

In [ ]:
## 08: 격자별 데이터가 존재하는 월 수 계산

month_counts = (
    df_visitor_summer_all
    .groupby('grid_id')['STD_YM']
    .nunique()
    .reset_index(name='matched_months_cnt')
)

grid_visitor_summer_avg = grid_visitor_summer_avg.merge(
    month_counts,
    on='grid_id',
    how='left'
)

In [ ]:
## 09: 격자 GeoDataFrame과 방문인구 데이터 결합

final_visitor_summer_gdf = gdf.merge(
    grid_visitor_summer_avg,
    on='grid_id',
    how='left'
)

In [ ]:
## 10: 최종 방문인구 데이터 저장

final_visitor_summer_gdf.to_csv(
    'final_visitor_summer_gdf.csv',
    index=False,
    encoding='utf-8-sig'
)

final_visitor_summer_gdf.to_file(
    'final_visitor_summer_gdf.shp',
    encoding='utf-8'
)

## 4. 주중 서비스인구 전처리 (09번 주중주말 서비스인구 中 주중)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geoband.API import *
GetCompasData('SBJ_2606_002', '9', '09._주중주말_서비스인구.csv')

import pandas as pd
df = pd.read_csv('09._주중주말_서비스인구.csv')
df.head()

df_weekday = df[df['hw'] == 'H'].copy()
df_weekend = df[df['hw'] == 'W'].copy()

print(f"📊 전체 데이터: {len(df):,}건")
print(f"🏢 주중(H) 데이터: {len(df_weekday):,}건")
print(f"🏖️ 주말(W) 데이터: {len(df_weekend):,}건\n")

print("--- 주중(H) 데이터 샘플 ---")
display(df_weekday.head())

print("--- 주말(W) 데이터 샘플 ---")
display(df_weekend.head())

df_weekday.to_csv('09._주중_서비스인구.csv', index=False, encoding='utf-8-sig')
df_weekend.to_csv('09._주말_서비스인구.csv', index=False, encoding='utf-8-sig')

print("✅ 주중/주말 데이터 분리 및 CSV 저장 완료!")

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.read_file('04._격자_30mx30m.geojson')

df_weekday = pd.read_csv('09._주중_서비스인구.csv')

geometry = [Point(xy) for xy in zip(df_weekday['lon'], df_weekday['lat'])]
weekday_gdf = gpd.GeoDataFrame(df_weekday, geometry=geometry, crs="EPSG:4326")

if gdf.crs != weekday_gdf.crs:
    gdf = gdf.to_crs(weekday_gdf.crs)

joined_weekday_df = gpd.sjoin(weekday_gdf, gdf[['grid_id', 'geometry']], how='left', predicate='within')

result_weekday_df = pd.DataFrame(joined_weekday_df.drop(columns=['geometry', 'index_right'], errors='ignore'))

display(result_weekday_df.head())

result_weekday_df.to_csv('09._주중_서비스인구_grid_mapped.csv', index=False, encoding='utf-8-sig')
print("✅ 주중 서비스인구 격자 매핑 및 CSV 저장 완료!")

matched_weekday_df = result_weekday_df[result_weekday_df['grid_id'].notna()]

total_rows = len(result_weekday_df)
matched_rows = len(matched_weekday_df)
match_rate = (matched_rows / total_rows) * 100

print(f"📊 전체 주중 서비스인구 데이터: {total_rows:,}건")
print(f"✅ 격자(grid_id) 매칭 성공: {matched_rows:,}건 ({match_rate:.2f}%)")
print(f"❌ 매칭 실패(영역 외): {total_rows - matched_rows:,}건\n")

print("--- 매칭 성공한 상위 5개 주중 서비스인구 데이터 ---")
display(matched_weekday_df.head())

df_weekday_202406_matched = result_weekday_df[
    (result_weekday_df['STD_YM'].astype(str).str.contains('202406')) &
    (result_weekday_df['grid_id'].notna())
]
print(f"✅ 2024년 6월 분당구 주중 서비스인구 매칭 성공 데이터: {len(df_weekday_202406_matched):,}개")
display(df_weekday_202406_matched.head())

df_weekday_202407_matched = result_weekday_df[
    (result_weekday_df['STD_YM'].astype(str).str.contains('202407')) &
    (result_weekday_df['grid_id'].notna())
]
print(f"✅ 2024년 7월 분당구 주중 서비스인구 매칭 성공 데이터: {len(df_weekday_202407_matched):,}개")
display(df_weekday_202407_matched.head())

df_weekday_202408_matched = result_weekday_df[
    (result_weekday_df['STD_YM'].astype(str).str.contains('202408')) &
    (result_weekday_df['grid_id'].notna())
]
print(f"✅ 2024년 8월 분당구 주중 서비스인구 매칭 성공 데이터: {len(df_weekday_202408_matched):,}개")
display(df_weekday_202408_matched.head())

df_weekday_202409_matched = result_weekday_df[
    (result_weekday_df['STD_YM'].astype(str).str.contains('202409')) &
    (result_weekday_df['grid_id'].notna())
]
print(f"✅ 2024년 9월 분당구 주중 서비스인구 매칭 성공 데이터: {len(df_weekday_202409_matched):,}개")
display(df_weekday_202409_matched.head())

import pandas as pd
import geopandas as gpd

df_weekday_summer_all = pd.concat([
    df_weekday_202406_matched,
    df_weekday_202407_matched,
    df_weekday_202408_matched,
    df_weekday_202409_matched
], ignore_index=True)

target_cols = ['w_pop', 'v_pop']

grid_weekday_summer_avg = df_weekday_summer_all.groupby('grid_id')[target_cols].mean().reset_index()

month_counts = df_weekday_summer_all.groupby('grid_id')['STD_YM'].nunique().reset_index(name='matched_months_cnt')
grid_weekday_summer_avg = grid_weekday_summer_avg.merge(month_counts, on='grid_id', how='left')

final_weekday_summer_gdf = gdf.merge(grid_weekday_summer_avg, on='grid_id', how='left')

display(final_weekday_summer_gdf[['grid_id', 'matched_months_cnt'] + target_cols].dropna().head())

final_weekday_summer_gdf.to_file('final_weekday_summer_gdf.shp', encoding='utf-8')
final_weekday_summer_gdf.to_csv('final_weekday_summer_gdf.csv', index=False, encoding='utf-8-sig')

print("✅ 6~9월 주중 서비스인구 가변 달 수 기준 평균 매핑 및 파일 저장 완료!")

## 5. 주말 서비스인구 전처리 (09번 주중주말 서비스인구 中 주말)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geoband.API import *
GetCompasData('SBJ_2606_002', '9', '09._주중주말_서비스인구.csv')

import pandas as pd
df = pd.read_csv('09._주중주말_서비스인구.csv')
df.head()

df_weekday = df[df['hw'] == 'H'].copy()
df_weekend = df[df['hw'] == 'W'].copy()

print(f"📊 전체 데이터: {len(df):,}건")
print(f"🏢 주중(H) 데이터: {len(df_weekday):,}건")
print(f"🏖️ 주말(W) 데이터: {len(df_weekend):,}건\n")

print("--- 주중(H) 데이터 샘플 ---")
display(df_weekday.head())

print("--- 주말(W) 데이터 샘플 ---")
display(df_weekend.head())

df_weekday.to_csv('09._주중_서비스인구.csv', index=False, encoding='utf-8-sig')
df_weekend.to_csv('09._주말_서비스인구.csv', index=False, encoding='utf-8-sig')

print("✅ 주중/주말 데이터 분리 및 CSV 저장 완료!")

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.read_file('04._격자_30mx30m.geojson')

df_weekend = pd.read_csv('09._주말_서비스인구.csv')

geometry = [Point(xy) for xy in zip(df_weekend['lon'], df_weekend['lat'])]
weekend_gdf = gpd.GeoDataFrame(df_weekend, geometry=geometry, crs="EPSG:4326")

if gdf.crs != weekend_gdf.crs:
    gdf = gdf.to_crs(weekend_gdf.crs)

joined_weekend_df = gpd.sjoin(weekend_gdf, gdf[['grid_id', 'geometry']], how='left', predicate='within')

result_weekend_df = pd.DataFrame(joined_weekend_df.drop(columns=['geometry', 'index_right'], errors='ignore'))

display(result_weekend_df.head())

result_weekend_df.to_csv('09._주말_서비스인구_grid_mapped.csv', index=False, encoding='utf-8-sig')
print("✅ 주말 서비스인구 격자 매핑 및 CSV 저장 완료!")

matched_weekend_df = result_weekend_df[result_weekend_df['grid_id'].notna()]

total_rows = len(result_weekend_df)
matched_rows = len(matched_weekend_df)
match_rate = (matched_rows / total_rows) * 100

print(f"📊 전체 주말 서비스인구 데이터: {total_rows:,}건")
print(f"✅ 격자(grid_id) 매칭 성공: {matched_rows:,}건 ({match_rate:.2f}%)")
print(f"❌ 매칭 실패(영역 외): {total_rows - matched_rows:,}건\n")

print("--- 매칭 성공한 상위 5개 주말 서비스인구 데이터 ---")
display(matched_weekend_df.head())

df_weekend_202406_matched = result_weekend_df[
    (result_weekend_df['STD_YM'].astype(str).str.contains('202406')) &
    (result_weekend_df['grid_id'].notna())
]
print(f"✅ 2024년 6월 분당구 주말 서비스인구 매칭 성공 데이터: {len(df_weekend_202406_matched):,}개")
display(df_weekend_202406_matched.head())

df_weekend_202407_matched = result_weekend_df[
    (result_weekend_df['STD_YM'].astype(str).str.contains('202407')) &
    (result_weekend_df['grid_id'].notna())
]
print(f"✅ 2024년 7월 분당구 주말 서비스인구 매칭 성공 데이터: {len(df_weekend_202407_matched):,}개")
display(df_weekend_202407_matched.head())

df_weekend_202408_matched = result_weekend_df[
    (result_weekend_df['STD_YM'].astype(str).str.contains('202408')) &
    (result_weekend_df['grid_id'].notna())
]
print(f"✅ 2024년 8월 분당구 주말 서비스인구 매칭 성공 데이터: {len(df_weekend_202408_matched):,}개")
display(df_weekend_202408_matched.head())

df_weekend_202409_matched = result_weekend_df[
    (result_weekend_df['STD_YM'].astype(str).str.contains('202409')) &
    (result_weekend_df['grid_id'].notna())
]
print(f"✅ 2024년 9월 분당구 주말 서비스인구 매칭 성공 데이터: {len(df_weekend_202409_matched):,}개")
display(df_weekend_202409_matched.head())

import pandas as pd
import geopandas as gpd

df_weekend_summer_all = pd.concat([
    df_weekend_202406_matched,
    df_weekend_202407_matched,
    df_weekend_202408_matched,
    df_weekend_202409_matched
], ignore_index=True)

target_cols = ['w_pop', 'v_pop']

grid_weekend_summer_avg = df_weekend_summer_all.groupby('grid_id')[target_cols].mean().reset_index()

month_counts = df_weekend_summer_all.groupby('grid_id')['STD_YM'].nunique().reset_index(name='matched_months_cnt')
grid_weekend_summer_avg = grid_weekend_summer_avg.merge(month_counts, on='grid_id', how='left')

final_weekend_summer_gdf = gdf.merge(grid_weekend_summer_avg, on='grid_id', how='left')

display(final_weekend_summer_gdf[['grid_id', 'matched_months_cnt'] + target_cols].dropna().head())

final_weekend_summer_gdf.to_file('final_weekend_summer_gdf.shp', encoding='utf-8')
final_weekend_summer_gdf.to_csv('final_weekend_summer_gdf.csv', index=False, encoding='utf-8-sig')

print("✅ 6~9월 주말 서비스인구 가변 달 수 기준 평균 매핑 및 파일 저장 완료!")

## 6. 인구 데이터 통합 (population_master 생성)

위 5개 결과(shp)를 grid_id 기준으로 outer merge하고, 폭염 대응 관련 파생변수(`pop_peak_heat`, `pop_elderly`, `pop_lunch_worker`)를 생성한다. 중간 검증용 출력은 생략하고 핵심 결과만 표시한다.

In [ ]:
import pandas as pd
import geopandas as gpd
from functools import reduce

final_summer_gdf = gpd.read_file('summer_flow_population.shp')          # 유동인구
final_worker_summer_gdf = gpd.read_file('final_worker_summer_gdf.shp')   # 직장인구
final_visitor_summer_gdf = gpd.read_file('final_visitor_summer_gdf.shp') # 방문인구
final_weekday_summer_gdf = gpd.read_file('final_weekday_summer_gdf.shp') # 주중 서비스인구
final_weekend_summer_gdf = gpd.read_file('final_weekend_summer_gdf.shp') # 주말 서비스인구


In [ ]:
# 결측치 제거 (각 데이터셋의 대표 값 컬럼 기준)
val_col_1 = [c for c in final_summer_gdf.columns if c not in ['grid_id', 'geometry']][0]
final_summer_gdf = final_summer_gdf.dropna(subset=[val_col_1]).reset_index(drop=True)

val_col_2 = [c for c in final_worker_summer_gdf.columns if c not in ['grid_id', 'geometry']][0]
final_worker_summer_gdf = final_worker_summer_gdf.dropna(subset=[val_col_2]).reset_index(drop=True)

val_col_3 = [c for c in final_visitor_summer_gdf.columns if c not in ['grid_id', 'geometry']][0]
final_visitor_summer_gdf = final_visitor_summer_gdf.dropna(subset=[val_col_3]).reset_index(drop=True)

val_col_4 = [c for c in final_weekday_summer_gdf.columns if c not in ['grid_id', 'geometry']][0]
final_weekday_summer_gdf = final_weekday_summer_gdf.dropna(subset=[val_col_4]).reset_index(drop=True)

val_col_5 = [c for c in final_weekend_summer_gdf.columns if c not in ['grid_id', 'geometry']][0]
final_weekend_summer_gdf = final_weekend_summer_gdf.dropna(subset=[val_col_5]).reset_index(drop=True)

# grid_id 교차 검증 (핵심 결과만 출력)
set_summer = set(final_summer_gdf['grid_id'])
set_worker = set(final_worker_summer_gdf['grid_id'])
set_visitor = set(final_visitor_summer_gdf['grid_id'])
set_weekday = set(final_weekday_summer_gdf['grid_id'])
set_weekend = set(final_weekend_summer_gdf['grid_id'])

is_all_equal = (set_summer == set_worker == set_visitor == set_weekday == set_weekend)
print("grid_id 일치 여부:", "✅ 5개 데이터셋 100% 일치" if is_all_equal else "⚠️ 데이터셋 간 grid_id 불일치 존재")


In [ ]:
# 격자 geometry 통합 (5개 데이터셋 합집합 기준)
all_gdfs = [final_summer_gdf, final_worker_summer_gdf, final_visitor_summer_gdf,
            final_weekday_summer_gdf, final_weekend_summer_gdf]

grid_geom_list = [gdf[['grid_id', 'geometry']] for gdf in all_gdfs
                   if 'geometry' in gdf.columns and 'grid_id' in gdf.columns]
grid_geom_df = pd.concat(grid_geom_list, ignore_index=True).drop_duplicates(subset=['grid_id']).reset_index(drop=True)

# 데이터셋별 정리 및 컬럼명 표준화
df_flow = final_summer_gdf.drop(columns=['geometry'], errors='ignore').copy()
df_flow = df_flow.drop(columns=[c for c in df_flow.columns if 'matched' in c], errors='ignore')

df_worker = final_worker_summer_gdf.drop(columns=['geometry'], errors='ignore').copy()
df_worker = df_worker.drop(columns=[c for c in df_worker.columns if 'matched' in c], errors='ignore')
worker_renames = {col: f"work_{col.replace('TMST_', '')}h" for col in df_worker.columns if col.startswith('TMST_')}
df_worker.rename(columns=worker_renames, inplace=True)

df_visitor = final_visitor_summer_gdf.drop(columns=['geometry'], errors='ignore').copy()
df_visitor = df_visitor.drop(columns=[c for c in df_visitor.columns if 'matched' in c], errors='ignore')
visitor_renames = {col: f"visi_{col.replace('TMST_', '')}h" for col in df_visitor.columns if col.startswith('TMST_')}
df_visitor.rename(columns=visitor_renames, inplace=True)

df_weekday = final_weekday_summer_gdf.drop(columns=['geometry'], errors='ignore').copy()
df_weekday.rename(columns={'w_pop': 'wd_w_pop', 'v_pop': 'wd_v_pop'}, inplace=True)
df_weekday = df_weekday[[c for c in ['grid_id', 'wd_w_pop', 'wd_v_pop'] if c in df_weekday.columns]]

df_weekend = final_weekend_summer_gdf.drop(columns=['geometry'], errors='ignore').copy()
df_weekend.rename(columns={'w_pop': 'we_w_pop', 'v_pop': 'we_v_pop'}, inplace=True)
df_weekend = df_weekend[[c for c in ['grid_id', 'we_w_pop', 'we_v_pop'] if c in df_weekend.columns]]

# grid_id 기준 outer merge
data_list = [df_flow, df_worker, df_visitor, df_weekday, df_weekend]
master_df = reduce(lambda left, right: pd.merge(left, right, on='grid_id', how='outer'), data_list)

pop_cols = [c for c in master_df.columns if c != 'grid_id']
master_df[pop_cols] = master_df[pop_cols].fillna(0)

# 폭염 대응 파생변수 생성
heat_hours = [f'visi_{h:02d}h' for h in range(12, 17)]
master_df['pop_peak_heat'] = master_df[heat_hours].mean(axis=1)          # 낮 12~16시 평균 방문인구

elderly_cols = [c for c in master_df.columns if '60g_pop' in c]
master_df['pop_elderly'] = master_df[elderly_cols].sum(axis=1)            # 60대 이상 인구 합계

master_df['pop_lunch_worker'] = master_df[['work_12h', 'work_13h']].mean(axis=1)  # 점심시간 직장인구

# 최종 GeoDataFrame 생성 및 저장
master_gdf = pd.merge(grid_geom_df, master_df, on='grid_id', how='left')
master_gdf = gpd.GeoDataFrame(master_gdf, geometry='geometry')

master_gdf.to_csv('population_master.csv', index=False, encoding='utf-8-sig')
master_gdf.to_file('population_master.shp', driver='ESRI Shapefile', encoding='utf-8')

print(f"✅ 인구 데이터 통합 완료 — 격자 수: {len(master_gdf):,}개, 컬럼 수: {len(master_gdf.columns)}개")
display(master_gdf.head(3))
